# Sarcasm Detection — Colab runner

Runs the whole project (TF-IDF baseline + BERT + RoBERTa, each **with** and **without** thread context), prints the comparison table, and saves the results to Drive.

By default it uses a **200k subsample, 1 epoch** — a complete, valid run that finishes in **~30–45 min** on a fast GPU. To train on the full ~1M dataset instead (several hours), see the commented alternative in step 4.

## Before you start
1. **Runtime → Change runtime type → GPU.**
2. Put **`train-balanced-sarcasm.csv`** somewhere in your Google Drive — you'll point to it in step 3. Get it from <https://www.kaggle.com/datasets/danofer/sarcasm>.

Then **Runtime → Run all**. With Colab Pro background execution it keeps running even if you close the tab.

In [ ]:
# 1. Which GPU did Colab give us?
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Get the code and install dependencies.
#    cd /content + rm first, so re-running this cell can't nest the clone inside a
#    previous copy.
%cd /content
!rm -rf SarcasmDetection
!git clone https://github.com/jan-stochnialek/SarcasmDetection.git
%cd /content/SarcasmDetection/project
!pip install -q -r requirements.txt

In [ ]:
# 3. Make the data available on FAST LOCAL DISK.
#    Reading a big CSV repeatedly over the Google Drive mount is unreliable (it can
#    fail mid-file with 'EOF inside string'), and the project reads it once per model
#    — so copy it to /content once and point the loader there.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_CSV = '/content/drive/MyDrive/sarcasm/train-balanced-sarcasm.csv'   # EDIT if needed
LOCAL_CSV = '/content/train-balanced-sarcasm.csv'
assert os.path.exists(DRIVE_CSV), 'CSV not found in Drive — fix DRIVE_CSV above.'
shutil.copy(DRIVE_CSV, LOCAL_CSV)
os.environ['SARC_CSV'] = LOCAL_CSV
print('Using local copy:', LOCAL_CSV, f'({os.path.getsize(LOCAL_CSV) / 1e6:.0f} MB, expect ~250)')

# Alternative — download with the Kaggle API straight to local disk (also grabs
# comments.json for the official test split). Upload kaggle.json first, then uncomment:
# !pip install -q kaggle && mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d danofer/sarcasm -p /content/sarc --unzip
# os.environ['SARC_CSV'] = '/content/sarc/train-balanced-sarcasm.csv'

In [ ]:
# 4. Batch size for this GPU (bf16 auto-on for Blackwell/A100/L4; a T4 falls back to fp16).
import torch
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
batch = 96 if mem_gb >= 70 else 48 if mem_gb >= 38 else 32 if mem_gb >= 22 else 16
print(f'{torch.cuda.get_device_name(0)}  ~{mem_gb:.0f} GB  ->  BATCH_SIZE = {batch}')
!sed -i 's/^BATCH_SIZE = .*/BATCH_SIZE = {batch}/' settings.py

# ----- ≤1 hour preset (complete, valid results) -----
# All 6 models on a 200k subsample, 1 epoch. Sequence length stays at 256 so the
# with-context condition isn't handicapped. Finishes in ~30–45 min on a fast GPU.
!sed -i 's/^SAMPLE_SIZE = .*/SAMPLE_SIZE = 200000/' settings.py
!sed -i 's/^EPOCHS = .*/EPOCHS = 1/' settings.py

# ----- Full-dataset alternative (best results, several hours) -----
# Comment the two lines above and uncomment these to use ALL ~1M comments:
# !sed -i 's/^SAMPLE_SIZE = .*/SAMPLE_SIZE = None/' settings.py
# !sed -i 's/^EPOCHS = .*/EPOCHS = 2/' settings.py

In [ ]:
# 5. Sanity-check the data loads, then run everything. The log is mirrored to Drive
#    so you keep it even if the runtime recycles.
!python check_data.py
!python run_everything.py 2>&1 | tee /content/drive/MyDrive/sarcasm/run.log

In [ ]:
# 6. Re-print the table and copy results (scores + confusion-matrix PNGs) to Drive.
!python show_results.py
!mkdir -p /content/drive/MyDrive/sarcasm/results && cp -r ../results/* /content/drive/MyDrive/sarcasm/results/
print('Saved results to Drive: MyDrive/sarcasm/results/')